In [3]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)
from pymilvus.model.hybrid import BGEM3EmbeddingFunction
from FlagEmbedding import BGEM3FlagModel
import torch

In [4]:
from pymilvus import connections, list_collections

connections.connect("default", host="localhost", port="19530")

if connections.has_connection("default"):
    print("Successfully connected to Milvus")
else:
    print("Failed to connect to Milvus")

# print all collections
print(list_collections())

Successfully connected to Milvus
['hybrid_dem3', 'hybrid_experiment_test2', 'sbert_experiment', 'hybrid_experiment_test', 'sbert_experiment_test', 'hybrid_experiment_test22']


In [5]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=1000),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=1024),  # Ensure the dimension matches your embeddings
    # Store sparse vectors
    FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),  
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'hybrid_experiment_test2'
col = Collection(col_name, schema, consistency_level="Strong")

In [6]:
sparse_index = {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"}
col.create_index("sparse_vector", sparse_index)
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
print("Created collection: ", col_name)
col.load()

Created collection:  hybrid_experiment_test2


In [7]:
ef = BGEM3EmbeddingFunction(use_fp16=True, device='cuda')

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

/home/sbasir/Thesis/myenv3.10/lib/python3.10/site-packages/FlagEmbedding/BGE_M3/modeling.py:335: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  colbert_state_dict = torch.loa

In [17]:
print(torch.cuda.memory_summary(device='cuda', abbreviated=True))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   1093 MiB |   3118 MiB |  11388 GiB |  11387 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   1093 MiB |   3118 MiB |  11388 GiB |  11387 GiB |
|---------------------------------------------------------------------------|
| Requested memory      |   1093 MiB |   3117 MiB |  11264 GiB |  11263 GiB |
|---------------------------------------------------------------

In [18]:
def merge_text_fields(data):
    for item in data:
        # Merge the fields into 'text', separating them by " | "
        merged_text = " | ".join(filter(None, [item.get('text', ''), 
                                            item.get('provided_data', ''), 
                                            item.get('enriched_data', ''), 
                                            item.get('translated_data', '')]))
        
        # Assign the merged text back to the 'text' field
        item['text'] = merged_text
        
        # Remove the individual fields as they're now part of 'text'
        item.pop('provided_data', None)
        item.pop('enriched_data', None)
        item.pop('translated_data', None)
    
    return data

def hybrid_embeddings(batch_data):
    data = merge_text_fields(batch_data)
    only_text = [x['text'] for x in data]

    # Generate embeddings using BGEM3 model
    embeddings = ef(only_text)
    # Prepare data for insertion
    
    # make sure all dense vectors are float32
    embeddings["dense"] = np.array(embeddings["dense"], dtype=np.float32)

    entities = [
        [item['id'] for item in data],  # IDs
        only_text,  # Texts
        embeddings["dense"],  # Dense vectors
        embeddings["sparse"]  # Sparse vectors
    ]

    del embeddings
    # torch.cuda.empty_cache()

    return entities

In [20]:
import os
import gzip
import json
import time
import tqdm
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed

docs_len_vals = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1000000]

# Function to load data from a compressed JSON file
def load_compressed_json(file_path):
    """Function to load data from a compressed JSON file."""
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        return json.load(f)

def load_data_in_batches_for_indexing(uncompressed_data, batch_size=100):
    """Index pre-loaded (uncompressed) data in batches for Milvus."""
    batch_data = []  # To store the current batch
    start_time = time.time()  # Record the start time
    total_docs_indexed = 0  # Track the total number of documents indexed

    checkpoint_times = {}  # Dictionary to save checkpoint times
    timestamp_file = f'timestamps_uncompressed_{batch_size}.txt'

    # Open the timestamp file in append mode instead of overwriting it
    with open(timestamp_file, 'a') as f_timestamp:

        # Loop over uncompressed data to process in batches
        for doc in tqdm.tqdm(uncompressed_data, desc="Batching and Indexing Data"):
            batch_data.append(doc)

            # Process in exact batches of 'batch_size' documents
            if len(batch_data) >= batch_size:
                # Call your Milvus insertion function here
                index_batch_into_milvus(batch_data[:batch_size])  # Process exactly 'batch_size' documents

                # Remove the processed documents from the batch
                del batch_data[:batch_size]

                # Update the total number of documents indexed
                total_docs_indexed += batch_size

                # Check if the total indexed documents exceed the next threshold in docs_len_vals
                while docs_len_vals and total_docs_indexed >= docs_len_vals[0]:
                    checkpoint_times[docs_len_vals[0]] = time.time() - start_time
                    print(f"Reached {docs_len_vals[0]} documents in {checkpoint_times[docs_len_vals[0]]:.2f} seconds.")
                    
                    # Append to the file to avoid overwriting
                    f_timestamp.write(f"Reached {docs_len_vals[0]} documents in {checkpoint_times[docs_len_vals[0]]:.2f} seconds.\n")
                    
                    docs_len_vals.pop(0)

        # Process any remaining documents
        if batch_data:
            while len(batch_data) > 0:
                if len(batch_data) >= batch_size:
                    # Process exactly 'batch_size' documents
                    index_batch_into_milvus(batch_data[:batch_size])
                    del batch_data[:batch_size]
                else:
                    # Process the remaining documents (less than batch_size)
                    index_batch_into_milvus(batch_data)
                    batch_data.clear()  # Clear the remaining batch after processing

    # Save the final checkpoint times to a file
    with open('checkpoint_times.json', 'w') as f:
        json.dump(checkpoint_times, f)

def index_batch_into_milvus(batch_data):
    """Function to index the batch data into Milvus."""
    entities = hybrid_embeddings(batch_data)
    col.insert(entities)
    del entities

def embed_and_index(parsed_directory, batch_size=100, max_workers=4):
    """Process one dataset at a time without loading all data at once."""
    # Collect all the .json.gz file paths
    file_paths = []
    for root, dirs, files in os.walk(parsed_directory):
        for file in files:
            if file.endswith('.json.gz'):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)
    
    total_files = len(file_paths)
    print(f"Found {total_files} files to process.")

    # Loop over each compressed file, uncompress, and process it
    for file_path in tqdm.tqdm(file_paths, desc="Processing files"):
        try:
            print(file_path)
            # Step 1: Uncompress the file and load its data
            uncompressed_data = load_compressed_json(file_path)
            
            # Step 2: Index the uncompressed data in batches
            load_data_in_batches_for_indexing(uncompressed_data, batch_size=batch_size)
            
            # Step 3: Delete uncompressed data after processing
            del uncompressed_data
            
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

In [21]:
# run the function to load data in batches
print(torch.cuda.memory_summary(device=None, abbreviated=False))
embed_and_index('/home/sbasir/Thesis/Thesis/cp2', batch_size=5, max_workers=1)
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   1093 MiB |   3118 MiB |  11388 GiB |  11387 GiB |
|       from large pool |   1092 MiB |   3117 MiB |  10775 GiB |  10774 GiB |
|       from small pool |      0 MiB |      6 MiB |    612 GiB |    612 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   1093 MiB |   3118 MiB |  11388 GiB |  11387 GiB |
|       from large pool |   1092 MiB |   3117 MiB |  10775 GiB |

Processing files:   0%|          | 0/11 [00:00<?, ?it/s]

/home/sbasir/Thesis/Thesis/cp2/113/113.json.gz


Reached 128 documents in 17.78 seconds.


Reached 256 documents in 36.02 seconds.


...

Reached 512 documents in 108.98 seconds.


Reached 1024 documents in 231.59 seconds.


Reached 2048 documents in 484.29 seconds.


Processing files:  18%|█▊        | 2/11 [12:01<1:02:38, 417.59s/it]

/home/sbasir/Thesis/Thesis/cp2/00101/00101.json.gz


In [8]:
query = "water"
query_embeddings = ef([query])

#make sure all dense vectors are float32
query_embeddings["dense"] = np.array(query_embeddings["dense"], dtype=np.float32)

k=100
# Prepare the search requests for both vector fields
sparse_search_params = {"metric_type": "IP"}
sparse_req = AnnSearchRequest(query_embeddings["sparse"],
                              "sparse_vector", sparse_search_params, limit=k)
dense_search_params = {"metric_type": "IP"}
dense_req = AnnSearchRequest(query_embeddings["dense"],
                             "dense_vector", dense_search_params, limit=k)

# Search topK docs based on dense and sparse vectors and rerank with RRF.
res = col.hybrid_search([sparse_req, dense_req], rerank=RRFRanker(),
                        limit=k, output_fields=['text'])

for result in res[0]:
    print(result)

id: /16/id_alba_p419384383, distance: 0.032786883413791656, entity: {'text': 'timestamp_update is 2023-09-12T12:42:44.208Z | type is IMAGE | content_tier is 4 | metadata_tier is B | edm:dataProvider is National Library of the Netherlands | edm:provider is National Library of the Netherlands | dc:creator is Jan Dhont | dc:description is Illustratie van een huis an de overkant van het water. Op het water zitten twee mannen in een boot. Aan de waterkant leunt een man met een hengel tegen een hek naast een wilgenboom. | fol. 26 | dc:language is und | dc:title is Albumbijdrage in het album amicorum van Jan Anthonij Snijders door J. Dhont | dc:type is Stammbücher (alba amicorum) | Albums amicorum | Stammbuch (album amicorum) | Album amicorum | alba amicorum | album amicorum | libri amicorum | liber amicorum | albums amicorum | albums amicorum | album amicorum | alba amicorum | album amicorum | Ill. Aquarel | Albuminscriptie | Album amicorum | dcterms:created is 1832/1844 | dcterms:medium is 